In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Gumawa ng mga Circuit

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.3.0
```
</details>
Tinitingnan ng pahinang ito nang mas malalim ang klase na [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) sa Qiskit SDK, kasama ang ilang mas advanced na pamamaraan na maaari mong gamitin para gumawa ng mga quantum circuit.
## Ano ang quantum circuit?
Ang isang simpleng quantum circuit ay isang koleksyon ng mga qubit at isang listahan ng mga instruksyon na kumikilos sa mga qubit na iyon. Para ipakita, ginagawa ng sumusunod na cell ang isang bagong circuit na may dalawang bagong qubit, pagkatapos ay ipinapakita ang attribute na [`qubits`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qubits) ng circuit, na isang listahan ng mga [`Qubit`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Qubit) mula sa pinakamaliit na significant bit na $q_0$ hanggang sa pinakamalaking significant bit na $q_n$.

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.qubits

[<Qubit register=(2, "q"), index=0>, <Qubit register=(2, "q"), index=1>]

Multiple `QuantumRegister` and `ClassicalRegister` objects can be combined to create a circuit. Every [`QuantumRegister`](/docs/api/qiskit/circuit#qiskit.circuit.QuantumRegister) and [`ClassicalRegister`](/docs/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) can also be named.

In [2]:
from qiskit.circuit import QuantumRegister, ClassicalRegister

qr1 = QuantumRegister(2, "qreg1")  # Create a QuantumRegister with 2 qubits
qr2 = QuantumRegister(1, "qreg2")  # Create a QuantumRegister with 1 qubit
cr1 = ClassicalRegister(3, "creg1")  # Create a ClassicalRegister with 3 cbits

combined_circ = QuantumCircuit(
    qr1, qr2, cr1
)  # Create a quantum circuit with 2 QuantumRegisters and 1 ClassicalRegister
combined_circ.qubits

[<Qubit register=(2, "qreg1"), index=0>,
 <Qubit register=(2, "qreg1"), index=1>,
 <Qubit register=(1, "qreg2"), index=0>]

Maraming `QuantumRegister` at `ClassicalRegister` na object ang maaaring pagsamahin para gumawa ng circuit. Ang bawat [`QuantumRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.QuantumRegister) at [`ClassicalRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) ay maaari ring pangalanan.

In [3]:
desired_qubit = qr2[0]  # Qubit 0 of register 'qreg2'

print("Index:", combined_circ.find_bit(desired_qubit).index)
print("Register:", combined_circ.find_bit(desired_qubit).registers)

Index: 2
Register: [(QuantumRegister(1, 'qreg2'), 0)]


Adding an instruction to the circuit appends the instruction to the circuit's [`data`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#data) attribute. The following cell output shows `data` is a list of [`CircuitInstruction`](/docs/api/qiskit/qiskit.circuit.CircuitInstruction) objects, each of which has an `operation` attribute, and a `qubits` attribute.

In [4]:
qc.x(0)  # Add X-gate to qubit 0
qc.data

[CircuitInstruction(operation=Instruction(name='x', num_qubits=1, num_clbits=0, params=[]), qubits=(<Qubit register=(2, "q"), index=0>,), clbits=())]

Mahahanap mo ang index at register ng isang qubit sa pamamagitan ng paggamit ng pamamaraang [`find_bit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.find_bit) ng circuit at ng mga attribute nito.

In [5]:
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg" alt="Output of the previous code cell" />

Circuit instruction objects can contain "definition" circuits that describe the instruction in terms of more fundamental instructions. For example, the [X-gate](/docs/api/qiskit/qiskit.circuit.library.XGate) is defined as a specific case of the [U3-gate](/docs/api/qiskit/qiskit.circuit.library.U3Gate), a more general single-qubit gate.

In [6]:
# Draw definition circuit of 0th instruction in `qc`
qc.data[0].operation.definition.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg" alt="Output of the previous code cell" />

Ang pagdaragdag ng instruksyon sa circuit ay nagdadagdag ng instruksyon sa attribute na [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#data) ng circuit. Ipinapakita ng output ng sumusunod na cell na ang `data` ay isang listahan ng mga object na [`CircuitInstruction`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.CircuitInstruction), na ang bawat isa ay may attribute na `operation` at attribute na `qubits`.

In [7]:
from qiskit.circuit.library import HGate

qc = QuantumCircuit(1)
qc.append(
    HGate(),  # New HGate instruction
    [0],  # Apply to qubit 0
)
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg" alt="Output of the previous code cell" />

To combine two circuits, use the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method. This accepts another [`QuantumCircuit`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit) and an optional list of qubit mappings.

<Admonition type="note">
    The [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method returns a new circuit and does **not** mutate either circuit it acts on. To mutate the circuit on which you're calling the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method, use the argument `inplace=True`.
</Admonition>

In [8]:
qc_a = QuantumCircuit(4)
qc_a.x(0)

qc_b = QuantumCircuit(2, name="qc_b")
qc_b.y(0)
qc_b.z(1)

# compose qubits (0, 1) of qc_a to qubits (1, 3) of qc_b respectively
combined = qc_a.compose(qc_b, qubits=[1, 3])
combined.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/29152dfa-2275-4bc4-aadb-82185b9e0e86-0.svg" alt="Output of the previous code cell" />

Ang pinakamadaling paraan para tingnan ang impormasyong ito ay sa pamamagitan ng pamamaraang [`draw`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#draw), na nagbabalik ng visualization ng circuit. Tingnan ang [Visualize circuits](./visualize-circuits) para sa iba't ibang paraan ng pagpapakita ng mga quantum circuit.

In [9]:
inst = qc_b.to_instruction()
qc_a.append(inst, [1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/81b682dd-45cb-4492-809e-d9e8ebbf5600-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg)

Ang mga object na circuit instruction ay maaaring maglaman ng mga "definition" na circuit na naglalarawan ng instruksyon sa pamamagitan ng mas pangunahing mga instruksyon. Halimbawa, ang [X-gate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.XGate) ay tinukoy bilang isang partikular na kaso ng [U3-gate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.U3Gate), isang mas pangkalahatang single-qubit gate.

In [10]:
gate = qc_b.to_gate().control()
qc_a.append(gate, [0, 1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg)

Ang mga instruksyon at circuit ay magkatulad dahil parehong naglalarawan ng mga operasyon sa mga bit at qubit, ngunit magkaiba ang kanilang mga layunin:

- Ang mga instruksyon ay itinuturing na naayos, at ang kanilang mga pamamaraan ay karaniwang nagbabalik ng mga bagong instruksyon (nang hindi binabago ang orihinal na object).
- Ang mga circuit ay idinisenyo para itayo sa maraming linya ng code, at ang mga pamamaraan ng [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) ay madalas na binabago ang kasalukuyang object.
### Ano ang circuit depth?
Ang [depth()](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.depth) ng isang quantum circuit ay isang sukatan ng bilang ng mga "layer" ng mga quantum gate, na isinasagawa nang sabay-sabay, na kailangan para matapos ang pagkalkula na tinukoy ng circuit. Dahil ang mga quantum gate ay nangangailangan ng oras para maipatupad, ang depth ng circuit ay halos katumbas ng dami ng oras na kailangan ng quantum computer para isagawa ang circuit. Kaya, ang depth ng circuit ay isa sa mga mahalagang dami na ginagamit para sukatin kung ang isang quantum circuit ay maaaring patakbuhin sa isang device.

Ipinapakita ng natitirang bahagi ng pahinang ito kung paano manipulahin ang mga quantum circuit.
## Gumawa ng mga Circuit
Ang mga pamamaraan tulad ng [`QuantumCircuit.h`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#h) at [`QuantumCircuit.cx`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#cx) ay nagdaragdag ng mga partikular na instruksyon sa mga circuit. Para magdagdag ng mga instruksyon sa circuit nang mas pangkalahatan, gamitin ang pamamaraang [`append`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#append). Tinatanggap nito ang isang instruksyon at isang listahan ng mga qubit na ilalapat ang instruksyon. Tingnan ang [Circuit Library API documentation](https://docs.quantum.ibm.com/api/qiskit/circuit_library) para sa listahan ng mga sinusuportahang instruksyon.

In [11]:
qc_a.decompose().draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg)

Para pagsamahin ang dalawang circuit, gamitin ang pamamaraang [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose). Tinatanggap nito ang isa pang [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) at isang opsyonal na listahan ng mga qubit mapping.

> **Note:** Ang pamamaraang [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose) ay nagbabalik ng bagong circuit at **hindi** binabago ang alinmang circuit na pinapatakbo nito. Para baguhin ang circuit kung saan mo tinatawag ang pamamaraang [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose), gamitin ang argument na `inplace=True`.

In [12]:
qc1 = QuantumCircuit(2, 2)
qc1.measure(0, 1)
qc1.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/0cdb2273-0.svg" alt="Output of the previous code cell" />

In [13]:
qc2 = QuantumCircuit(2)
qc2.measure_all()
qc2.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/6f33698c-0.svg" alt="Output of the previous code cell" />

In [14]:
qc3 = QuantumCircuit(2)
qc3.x(1)
qc3.measure_active()
qc3.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg)

Para makita kung ano ang nangyayari, maaari mong gamitin ang pamamaraang [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) para palawakin ang bawat instruksyon sa kahulugan nito.

> **Note:** Ang pamamaraang [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) ay nagbabalik ng bagong circuit at **hindi** binabago ang circuit na pinapatakbo nito.

In [15]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit import Parameter

angle = Parameter("angle")  # undefined number

# Create and optimize circuit once
qc = QuantumCircuit(1)
qc.rx(angle, 0)
qc = generate_preset_pass_manager(
    optimization_level=3, basis_gates=["u", "cx"]
).run(qc)

qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/a580552c-d585-4047-99f0-32aafd06e4f3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg)

<span id="measure-qubits"></span>
## Sukatin ang mga Qubit
Ginagamit ang mga measurement para sampulin ang mga estado ng mga indibidwal na qubit at ilipat ang mga resulta sa isang classical register. Tandaan na kung nagsusumite ka ng mga circuit sa isang primitive na [Sampler](./primitives#sampler), kailangan ang mga measurement. Gayunpaman, ang mga circuit na isinumite sa isang primitive na [Estimator](./primitives#estimator) ay hindi dapat maglaman ng mga measurement.

Maaaring sukatin ang mga qubit gamit ang tatlong pamamaraan: [`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.measure), [`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) at [`measure_active`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_active). Para malaman kung paano i-visualize ang mga nasukat na resulta, tingnan ang pahina na [Visualize results](./visualize-results).

1. `QuantumCircuit.measure` : sinusukat ang bawat qubit sa unang argument sa classical bit na ibinigay bilang pangalawang argument. Pinapayagan ng pamamaraang ito ang buong kontrol kung saan itatago ang resulta ng measurement.

2. `QuantumCircuit.measure_all` : walang tinatanggap na argument at maaaring gamitin para sa mga quantum circuit na walang paunang natukoy na mga classical bit. Gumagawa ito ng mga classical wire at iniimbak ang mga resulta ng measurement sa pagkakasunod-sunod. Halimbawa, ang measurement ng qubit na $q_i$ ay iniimbak sa cbit na $meas_i$). Nagdaragdag din ito ng barrier bago ang measurement.

3. `QuantumCircuit.measure_active` : katulad ng `measure_all`, ngunit sinusukat lamang ang mga qubit na may mga operasyon.

In [16]:
circuits = []
for value in range(100):
    circuits.append(qc.assign_parameters({angle: value}))

circuits[0].draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/85af6231-921a-4130-99d3-f6998f761df8-0.svg" alt="Output of the previous code cell" />

You can find a list of a circuit's undefined parameters in its `parameters` attribute.

In [17]:
qc.parameters

ParameterView([Parameter(angle)])

### Change a parameter's name

By default, parameter names for a parameterized circuit are prefixed by `x`- for example, `x[0]`. You can change the names after they are defined, as shown in the following example.

In [18]:
from qiskit.circuit.library import z_feature_map
from qiskit.circuit import ParameterVector

# Define a parameterized circuit with default names
# For example, x[0]
circuit = z_feature_map(2)

# Set new parameter names
# They will now be prefixed by `hi` instead
# For example, hi[0]
training_params = ParameterVector("hi", 2)

# Assign parameter names to the quantum circuit
circuit = circuit.assign_parameters(parameters=training_params)

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg)

## Mga parameterized na circuit

Maraming near-term quantum algorithm ang nangangailangan ng pagpapatakbo ng maraming variation ng isang quantum circuit. Dahil ang pagbuo at pag-optimize ng malalaking circuit ay maaaring mahal sa computational, sinusuportahan ng Qiskit ang mga **parameterized** na circuit. Ang mga circuit na ito ay may mga hindi natukoy na parameter, at hindi kailangang tukuyin ang kanilang mga halaga hanggang sa sandali bago isagawa ang circuit. Nagbibigay-daan ito sa iyo na ilipat ang pagbuo at pag-optimize ng circuit labas sa pangunahing program loop. Gumagawa at nagpapakita ang sumusunod na cell ng isang parameterized na circuit.